In [1]:
%pip install -q --root-user-action=ignore torch torchvision timm scikit-learn opencv-python pandas pillow torchxrayvision 

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import random

import cv2
import numpy as np
import pandas as pd

from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, classification_report, precision_recall_curve

import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchxrayvision as xrv

import timm
from timm.loss import AsymmetricLossMultiLabel

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message=".*iCCP.*")

def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    cudnn.deterministic = True
    cudnn.benchmark = False

set_seed(42)

In [3]:
import warnings
import logging
import os

# Silenciar warnings de Python
warnings.filterwarnings("ignore")

# Silenciar warnings de la librería de imágenes (libpng/PIL)
logging.getLogger("PIL").setLevel(logging.ERROR)

# Silenciar avisos de OpenCV y del sistema sobre metadatos
os.environ['OPENCV_LOG_LEVEL'] = 'OFF'

In [4]:
csv = "DATA/nih/Data_Entry_2017.csv"
imagenes_png_folder = "DATA/nih/data384"
df = pd.read_csv(csv)

In [5]:
os.environ["OPENCV_LOG_LEVEL"] = "SILENT"

In [6]:
df.head()

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,0.143,NaN
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,0.143,NaN
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168,NaN
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171,NaN
4,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143,0.143,NaN


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 112120 entries, 0 to 112119
Data columns (total 12 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Image Index                  112120 non-null  str    
 1   Finding Labels               112120 non-null  str    
 2   Follow-up #                  112120 non-null  int64  
 3   Patient ID                   112120 non-null  int64  
 4   Patient Age                  112120 non-null  int64  
 5   Patient Gender               112120 non-null  str    
 6   View Position                112120 non-null  str    
 7   OriginalImage[Width          112120 non-null  int64  
 8   Height]                      112120 non-null  int64  
 9   OriginalImagePixelSpacing[x  112120 non-null  float64
 10  y]                           112120 non-null  float64
 11  Unnamed: 11                  0 non-null       float64
dtypes: float64(3), int64(5), str(4)
memory usage: 13.7 MB


In [8]:
for col in df:
  print(df[col].value_counts(),"\n")

Image Index
00000001_000.png    1
00000001_001.png    1
00000001_002.png    1
00000002_000.png    1
00000003_000.png    1
                   ..
00030801_001.png    1
00030802_000.png    1
00030803_000.png    1
00030804_000.png    1
00030805_000.png    1
Name: count, Length: 112120, dtype: int64 

Finding Labels
No Finding                                                        60361
Infiltration                                                       9547
Atelectasis                                                        4215
Effusion                                                           3955
Nodule                                                             2705
                                                                  ...  
Atelectasis|Effusion|Emphysema|Pleural_Thickening|Pneumothorax        1
Consolidation|Pneumonia|Mass                                          1
Consolidation|Effusion|Emphysema                                      1
Atelectasis|Hernia|Infiltration|Mass|No

In [9]:
df['Finding Labels'].value_counts()

Finding Labels
No Finding                                                        60361
Infiltration                                                       9547
Atelectasis                                                        4215
Effusion                                                           3955
Nodule                                                             2705
                                                                  ...  
Atelectasis|Effusion|Emphysema|Pleural_Thickening|Pneumothorax        1
Consolidation|Pneumonia|Mass                                          1
Consolidation|Effusion|Emphysema                                      1
Atelectasis|Hernia|Infiltration|Mass|Nodule|Pneumothorax              1
Atelectasis|Consolidation|Mass|Pleural_Thickening|Pneumothorax        1
Name: count, Length: 836, dtype: int64

In [10]:
conteo_etiquetas = df['Finding Labels'].str.split('|').explode().value_counts()
porcentajes = df['Finding Labels'].str.split('|').explode().value_counts(normalize=True) * 100

tabla_resumen = pd.DataFrame({
    'Cantidad': conteo_etiquetas,
    'Porcentaje (%)': porcentajes.round(2)
})

tabla_resumen

,Cantidad,Porcentaje (%)
Finding Labels,,
No Finding,60361,42.65
Infiltration,19894,14.06
Effusion,13317,9.41
Atelectasis,11559,8.17
Nodule,6331,4.47
Mass,5782,4.09
Pneumothorax,5302,3.75
Consolidation,4667,3.30
Pleural_Thickening,3385,2.39


In [11]:
conteos_individuales = df['Finding Labels'].str.split('|').explode().value_counts()
clases_mayoritarias = set(conteos_individuales[conteos_individuales >= 5000].index.tolist())

df['Finding Labels'] = df['Finding Labels'].apply(
    lambda s: '|'.join(l for l in s.split('|') if l in clases_mayoritarias) or None
)
df = df[df['Finding Labels'].notna()].reset_index(drop=True)

# df = df[df['View Position'] == 'PA'].reset_index(drop=True)

NO_FINDING_FRAC = 0.30
RANDOM_STATE = 444

df_no_finding = df[df['Finding Labels'] == 'No Finding'].copy()
df_con_hallazgos = df[df['Finding Labels'] != 'No Finding'].copy()

print(f"No Finding antes: {len(df_no_finding)}")
print(f"Con hallazgos antes: {len(df_con_hallazgos)}")

df_no_finding = df_no_finding.sample(frac=NO_FINDING_FRAC, random_state=RANDOM_STATE)

df = pd.concat([df_con_hallazgos, df_no_finding], ignore_index=True)
df = df.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

print(f"No Finding despues: {len(df[df['Finding Labels'] == 'No Finding'])}")
print(f"Total despues del filtro drastico: {len(df)}")

No Finding antes: 60361
Con hallazgos antes: 44957
No Finding despues: 18108
Total despues del filtro drastico: 63065


In [12]:
conteo_etiquetas = df['Finding Labels'].str.split('|').explode().value_counts()
porcentajes = df['Finding Labels'].str.split('|').explode().value_counts(normalize=True) * 100

tabla_resumen = pd.DataFrame({
    'Cantidad': conteo_etiquetas,
    'Porcentaje (%)': porcentajes.round(2)
})

tabla_resumen

,Cantidad,Porcentaje (%)
Finding Labels,,
Infiltration,19894,24.78
No Finding,18108,22.55
Effusion,13317,16.59
Atelectasis,11559,14.40
Nodule,6331,7.88
Mass,5782,7.20
Pneumothorax,5302,6.60


In [13]:
conteos_individuales = df['Finding Labels'].str.split('|').explode().value_counts()
clases_mayoritarias = set(conteos_individuales[conteos_individuales >= 5000].index.tolist())

df['Finding Labels'] = df['Finding Labels'].apply(
    lambda s: '|'.join(l for l in s.split('|') if l in clases_mayoritarias) or None
)
df = df[df['Finding Labels'].notna()].reset_index(drop=True)

# df = df[df['View Position'] == 'PA'].reset_index(drop=True)

NO_FINDING_FRAC = 0.30
RANDOM_STATE = 444

df_no_finding = df[df['Finding Labels'] == 'No Finding'].copy()
df_con_hallazgos = df[df['Finding Labels'] != 'No Finding'].copy()

print(f"No Finding antes: {len(df_no_finding)}")
print(f"Con hallazgos antes: {len(df_con_hallazgos)}")

df_no_finding = df_no_finding.sample(frac=NO_FINDING_FRAC, random_state=RANDOM_STATE)

df = pd.concat([df_con_hallazgos, df_no_finding], ignore_index=True)
df = df.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

print(f"No Finding despues: {len(df[df['Finding Labels'] == 'No Finding'])}")
print(f"Total despues del filtro drastico: {len(df)}")

No Finding antes: 18108
Con hallazgos antes: 44957
No Finding despues: 5432
Total despues del filtro drastico: 50389


In [14]:
conteo_etiquetas = df['Finding Labels'].str.split('|').explode().value_counts()
porcentajes = df['Finding Labels'].str.split('|').explode().value_counts(normalize=True) * 100

tabla_resumen = pd.DataFrame({
    'Cantidad': conteo_etiquetas,
    'Porcentaje (%)': porcentajes.round(2)
})

tabla_resumen

,Cantidad,Porcentaje (%)
Finding Labels,,
Infiltration,19894,29.42
Effusion,13317,19.69
Atelectasis,11559,17.09
Nodule,6331,9.36
Mass,5782,8.55
No Finding,5432,8.03
Pneumothorax,5302,7.84


In [15]:
gss_test = GroupShuffleSplit(test_size=0.30, n_splits=1, random_state=444)
for train_val_idx, test_idx in gss_test.split(df, groups=df['Patient ID']):
    df_train_val = df.iloc[train_val_idx].reset_index(drop=True)
    df_test = df.iloc[test_idx].reset_index(drop=True)


gss_val = GroupShuffleSplit(test_size=0.30, n_splits=1, random_state=444)
for train_idx, val_idx in gss_val.split(df_train_val, groups=df_train_val['Patient ID']):
    df_train = df_train_val.iloc[train_idx].reset_index(drop=True)
    df_val = df_train_val.iloc[val_idx].reset_index(drop=True)

print("PARTICIÓN A NIVEL PACIENTE COMPLETADA")

print(f"Total DataSet: {len(df)} imágenes")
print(f" -> Train: {len(df_train)} imágenes ({len(df_train['Patient ID'].unique())} pacientes)")
print(f" -> Validación: {len(df_val)} imágenes ({len(df_val['Patient ID'].unique())} pacientes)")
print(f" -> Test: {len(df_test)} imágenes ({len(df_test['Patient ID'].unique())} pacientes)")

PARTICIÓN A NIVEL PACIENTE COMPLETADA
Total DataSet: 50389 imágenes
 -> Train: 24731 imágenes (7380 pacientes)
 -> Validación: 10229 imágenes (3164 pacientes)
 -> Test: 15429 imágenes (4520 pacientes)


In [16]:
train_images = df_train['Image Index'].tolist()
val_images = df_val['Image Index'].tolist()
test_images = df_test['Image Index'].tolist()

In [17]:
with open('DATA/nih/my_patient_train_list.txt', 'w') as f:
    f.write('\n'.join(train_images))
with open('DATA/nih/my_patient_val_list.txt', 'w') as f:
    f.write('\n'.join(val_images))
with open('DATA/nih/my_patient_test_list.txt', 'w') as f:
    f.write('\n'.join(test_images))

In [18]:
df_train_val.head()

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00000073_001.png,No Finding,1,73,62,F,PA,2048,2500,0.168,0.168,NaN
1,00010505_016.png,Atelectasis|Effusion,16,10505,46,F,PA,2910,2991,0.143,0.143,NaN
2,00016797_013.png,Effusion|Pneumothorax,13,16797,74,F,AP,2500,2048,0.168,0.168,NaN
3,00026565_000.png,Atelectasis|Effusion,0,26565,44,F,PA,2654,2701,0.143,0.143,NaN
4,00005832_013.png,Infiltration,13,5832,45,M,AP,2500,2048,0.168,0.168,NaN


In [19]:
df_test.head()

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00001634_006.png,Infiltration,6,1634,45,F,PA,2542,2549,0.143,0.143,NaN
1,00027818_000.png,Pneumothorax,0,27818,33,F,PA,2992,2991,0.143,0.143,NaN
2,00023325_037.png,Infiltration,37,23325,83,F,AP,2500,2048,0.168,0.168,NaN
3,00027464_004.png,No Finding,4,27464,34,F,AP,2788,2544,0.139,0.139,NaN
4,00005069_016.png,Infiltration,16,5069,29,F,AP,3056,2544,0.139,0.139,NaN


In [20]:
df_train.head()

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00000073_001.png,No Finding,1,73,62,F,PA,2048,2500,0.168,0.168,NaN
1,00010505_016.png,Atelectasis|Effusion,16,10505,46,F,PA,2910,2991,0.143,0.143,NaN
2,00016797_013.png,Effusion|Pneumothorax,13,16797,74,F,AP,2500,2048,0.168,0.168,NaN
3,00026565_000.png,Atelectasis|Effusion,0,26565,44,F,PA,2654,2701,0.143,0.143,NaN
4,00005832_013.png,Infiltration,13,5832,45,M,AP,2500,2048,0.168,0.168,NaN


In [21]:
df_val.head()

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00021711_014.png,Effusion|Infiltration,14,21711,71,M,PA,2992,2991,0.143,0.143,NaN
1,00025252_007.png,Effusion|Pneumothorax,7,25252,30,M,AP,3056,2544,0.139,0.139,NaN
2,00018126_001.png,Infiltration|Nodule,1,18126,30,M,PA,2992,2989,0.143,0.143,NaN
3,00005107_000.png,Nodule,0,5107,44,F,PA,2048,2500,0.171,0.171,NaN
4,00025949_000.png,Infiltration,0,25949,29,F,PA,2992,2991,0.143,0.143,NaN


In [22]:
pacientes_train = set(df_train['Patient ID'].unique())
pacientes_val = set(df_val['Patient ID'].unique())
pacientes_test = set(df_test['Patient ID'].unique())

train_vs_val = pacientes_train.intersection(pacientes_val)
train_vs_test = pacientes_train.intersection(pacientes_test)
val_vs_test = pacientes_val.intersection(pacientes_test)

print(f"Pacientes en común entre Train y Val:  {len(train_vs_val)}")
print(f"Pacientes en común entre Train y Test: {len(train_vs_test)}")
print(f"Pacientes en común entre Val y Test:   {len(val_vs_test)}")

Pacientes en común entre Train y Val:  0
Pacientes en común entre Train y Test: 0
Pacientes en común entre Val y Test:   0


In [23]:
enfermedades = [
    "Atelectasis", "Effusion", "Infiltration", "Mass", "Nodule", "Pneumothorax"
]

# Precomputar etiquetas para el conjunto de entrenamiento
labels_train_df = (
    df_train["Finding Labels"]
    .str.get_dummies(sep="|")
    .reindex(columns=enfermedades, fill_value=0)
)

positivos = torch.tensor(labels_train_df.sum(axis=0).values, dtype=torch.float32)
negativos = torch.tensor(len(df_train), dtype=torch.float32) - positivos

bias_tensor = torch.log(positivos / negativos)


print("Negativos por clase:", dict(zip(enfermedades, negativos.tolist())))
print("Positivos por clase:", dict(zip(enfermedades, positivos.tolist())))
print("Biases iniciales:", bias_tensor.tolist())

Negativos por clase: {'Atelectasis': 19012.0, 'Effusion': 18216.0, 'Infiltration': 14914.0, 'Mass': 21918.0, 'Nodule': 21671.0, 'Pneumothorax': 22229.0}
Positivos por clase: {'Atelectasis': 5719.0, 'Effusion': 6515.0, 'Infiltration': 9817.0, 'Mass': 2813.0, 'Nodule': 3060.0, 'Pneumothorax': 2502.0}
Biases iniciales: [-1.201276421546936, -1.0281931161880493, -0.4181848168373108, -2.0530567169189453, -1.9575600624084473, -2.184307336807251]


In [24]:
class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, img_dir, enfermedades, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.enfermedades = enfermedades
        self.image_names = self.dataframe['Image Index'].values

        etiquetas_list = []
        for enf in enfermedades:
            if isinstance(self.dataframe['Finding Labels'].iloc[0], list):
                col = self.dataframe['Finding Labels'].apply(lambda x: 1.0 if enf in x else 0.0)
            else:
                col = self.dataframe['Finding Labels'].str.contains(enf, regex=False).astype(float)
            etiquetas_list.append(col.values)
            
        self.labels = torch.tensor(np.column_stack(etiquetas_list), dtype=torch.float32)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        img_path = os.path.join(self.img_dir, img_name)
        
        image = cv2.imread(img_path)
        if image is None:
            image = np.zeros((224, 224, 3), dtype=np.uint8)
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
        if self.transform:
            image = self.transform(image)
            
        return image, self.labels[idx]

In [25]:
class ApplyCLAHE(object):
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, img):
        if not isinstance(img, np.ndarray):
            img = np.array(img)
            
        img_yuv = cv2.cvtColor(img, cv2.COLOR_RGB2YUV)
        clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size)
        img_yuv[:, :, 0] = clahe.apply(img_yuv[:, :, 0])
        img_rgb = cv2.cvtColor(img_yuv, cv2.COLOR_YUV2RGB)
        
        return Image.fromarray(img_rgb)

In [26]:
import gc
# Limpieza preventiva para vaciar cualquier rastro del Swin Transformer
gc.collect()
torch.cuda.empty_cache()

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]
BATCH_SIZE = 32

# --- PIPELINES DE TRANSFORMACIÓN (Regresamos a 384px) ---
transformaciones_entrenamiento = transforms.Compose([
    ApplyCLAHE(clip_limit=2.0, tile_grid_size=(8, 8)),
    transforms.ToPILImage(),
    transforms.Resize((384, 384)), # <-- Resolución restaurada para ver nódulos
    transforms.RandomAffine(degrees=6, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.RandomHorizontalFlip(p=0.4),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

transformaciones_validacion = transforms.Compose([
    ApplyCLAHE(clip_limit=2.0, tile_grid_size=(8, 8)),
    transforms.ToPILImage(),
    transforms.Resize((384, 384)), # <-- Resolución restaurada
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

# Actualizar los datasets con las nuevas transformaciones
train_dataset.transform = transformaciones_entrenamiento
val_dataset.transform = transformaciones_validacion
test_dataset.transform = transformaciones_validacion

# --- DATALOADERS (4 workers activos y seguros para ConvNeXt) ---
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=4, persistent_workers=True, prefetch_factor=2, pin_memory=True
)

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, persistent_workers=False, prefetch_factor=2, pin_memory=True
)

test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, persistent_workers=False, prefetch_factor=2, pin_memory=True
)

In [27]:
train_dataset = ChestXrayDataset(
    dataframe=df_train,
    img_dir=imagenes_png_folder,
    enfermedades=enfermedades,
    transform=transformaciones_entrenamiento
)

print(len(train_dataset))

24731


In [28]:
val_dataset = ChestXrayDataset(
    dataframe=df_val,
    img_dir=imagenes_png_folder,
    enfermedades=enfermedades,
    transform=transformaciones_validacion
)

test_dataset = ChestXrayDataset(
    dataframe=df_test,
    img_dir=imagenes_png_folder,
    enfermedades=enfermedades,
    transform=transformaciones_validacion
)

In [29]:
import gc
import torch

# Limpieza preventiva de memoria y procesos antiguos
gc.collect()
torch.cuda.empty_cache()

BATCH_SIZE = 32

# Volvemos a tu configuración de 4 workers que te funcionaba bien
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,           # <--- Restaurado a 4
    persistent_workers=True, # <--- Restaurado
    prefetch_factor=2,       # <--- Restaurado
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False,
    num_workers=4, 
    persistent_workers=False, 
    prefetch_factor=2, 
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False,
    num_workers=4, 
    persistent_workers=False, 
    prefetch_factor=2, 
    pin_memory=True
)

In [30]:
class ChestXrayExpert(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = timm.create_model("swin_tiny_patch4_window7_224", pretrained=True, num_classes=num_classes)

    def forward(self, x):
        return self.backbone(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelo_experto = ChestXrayExpert(num_classes=len(enfermedades))

bias_inicial = []
for enf in enfermedades:
    if isinstance(df_train['Finding Labels'].iloc[0], list):
        positivos = df_train['Finding Labels'].apply(lambda x: enf in x).sum()
    else:
        positivos = df_train['Finding Labels'].str.contains(enf, regex=False).sum()
    negativos = len(df_train) - positivos
    bias_inicial.append(np.log(positivos / (negativos + 1e-5)))

with torch.no_grad():
    modelo_experto.backbone.head.fc.bias.copy_(torch.tensor(bias_inicial, dtype=torch.float32))

modelo_experto = modelo_experto.to(device)
if torch.cuda.device_count() > 1:
    modelo_experto = nn.DataParallel(modelo_experto)

In [31]:
modelo_experto.backbone.requires_grad_(False)
modelo_experto.backbone.head.requires_grad_(True)
trainable = sum(p.numel() for p in modelo_experto.parameters() if p.requires_grad)
print(f"Fase 1 activa — Parámetros entrenables: {trainable:,}")

Fase 1 activa — Parámetros entrenables: 4,614


In [32]:
'''import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- NUEVA FUNCIÓN DE PÉRDIDA: FOCAL LOSS ---
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt)**self.gamma * bce_loss
        return focal_loss.mean()

criterion = FocalLoss(alpha=0.25, gamma=2.0)
# --------------------------------------------

for param in modelo_experto.parameters():
    param.requires_grad = True

if hasattr(modelo_experto, 'module'):
    parametros_cabeza = list(modelo_experto.module.backbone.head.fc.parameters())
    parametros_base = [p for n, p in modelo_experto.module.backbone.named_parameters() if not n.startswith('head.fc')]
else:
    parametros_cabeza = list(modelo_experto.backbone.head.fc.parameters())
    parametros_base = [p for n, p in modelo_experto.backbone.named_parameters() if not n.startswith('head.fc')]

optimizer = torch.optim.AdamW([
    {'params': parametros_base, 'lr': 1e-5},
    {'params': parametros_cabeza, 'lr': 1e-4}
], weight_decay=0.05)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=5,
    T_mult=2,
    eta_min=1e-6
)

scaler = torch.amp.GradScaler('cuda', enabled=(device.type == "cuda"))

NUM_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 20
mejor_f1 = 0.0
epocas_sin_mejora = 0'''

'import torch.nn.functional as F\n\ndevice = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n\n# --- NUEVA FUNCIÓN DE PÉRDIDA: FOCAL LOSS ---\nclass FocalLoss(nn.Module):\n    def __init__(self, alpha=0.25, gamma=2.0):\n        super(FocalLoss, self).__init__()\n        self.alpha = alpha\n        self.gamma = gamma\n\n    def forward(self, inputs, targets):\n        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction=\'none\')\n        pt = torch.exp(-bce_loss)\n        focal_loss = self.alpha * (1 - pt)**self.gamma * bce_loss\n        return focal_loss.mean()\n\ncriterion = FocalLoss(alpha=0.25, gamma=2.0)\n# --------------------------------------------\n\nfor param in modelo_experto.parameters():\n    param.requires_grad = True\n\nif hasattr(modelo_experto, \'module\'):\n    parametros_cabeza = list(modelo_experto.module.backbone.head.fc.parameters())\n    parametros_base = [p for n, p in modelo_experto.module.backbone.named_parameters() if not n.s

In [33]:
from timm.data import Mixup

# Configuración de Mixup y CutMix
mixup_fn = Mixup(
    mixup_alpha=0.8,       # Intensidad de Mixup
    cutmix_alpha=1.0,      # Intensidad de CutMix
    prob=0.5,              # 50% de probabilidad de aplicar uno u otro
    switch_prob=0.5, 
    mode='batch',
    label_smoothing=0.1,   # Ayuda a que el modelo no se sobre-ajuste
    num_classes=len(enfermedades)
)

In [36]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import timm
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, classification_report, precision_recall_curve

# 1. CONFIGURACIÓN INICIAL
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 20
mejor_f1 = 0.0
epocas_sin_mejora = 0

# 2. DEFINICIÓN DEL MODELO (Swin Transformer)
class ChestXrayExpert(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # Usamos swin_tiny para que quepa en tus 12GB de VRAM
        self.backbone = timm.create_model("swin_tiny_patch4_window7_224", pretrained=True, num_classes=num_classes)

    def forward(self, x):
        return self.backbone(x)

modelo_experto = ChestXrayExpert(num_classes=len(enfermedades)).to(device)

# 3. FUNCIÓN DE PÉRDIDA (Focal Loss para clases difíciles)
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt)**self.gamma * bce_loss
        return focal_loss.mean()

criterion = FocalLoss(alpha=0.25, gamma=2.0)

# 4. OPTIMIZADOR Y SCHEDULER (Tasas diferenciales)
# Separamos la cabeza del cuerpo para un entrenamiento más fino
parametros_cabeza = list(modelo_experto.backbone.head.parameters())
parametros_base = [p for n, p in modelo_experto.backbone.named_parameters() if not n.startswith('head')]

optimizer = torch.optim.AdamW([
    {'params': parametros_base, 'lr': 1e-5},
    {'params': parametros_cabeza, 'lr': 1e-4}
], weight_decay=0.05)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == "cuda"))

# ========================================================
# 5. BUCLE DE ENTRENAMIENTO PRINCIPAL
# ========================================================
for epoch in range(NUM_EPOCHS):
    
    # Lógica de desbloqueo progresivo
    if epoch == 10:
        optimizer.param_groups[0]["lr"] = 1e-4 # Subimos velocidad al backbone
        print("\n>>> FASE 2: Backbone liberado para aprendizaje profundo")

    # --- ENTRENAMIENTO ---
    modelo_experto.train()
    perdida_train = 0.0
    loop_train = tqdm(train_loader, desc=f"Época {epoch+1}/{NUM_EPOCHS} [Train]", leave=False)

    for imagenes, etiquetas in loop_train:
        imagenes, etiquetas = imagenes.to(device, non_blocking=True), etiquetas.to(device, non_blocking=True)

        # NATIVE MULTI-LABEL MIXUP (50% prob)
        if np.random.rand() < 0.5:
            lam = np.random.beta(0.8, 0.8)
            index = torch.randperm(imagenes.size(0)).to(device)
            imagenes = lam * imagenes + (1 - lam) * imagenes[index, :]
            etiquetas = lam * etiquetas + (1 - lam) * etiquetas[index, :]

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", dtype=torch.float16, enabled=(device.type == "cuda")):
            salidas = modelo_experto(imagenes)
            loss = criterion(salidas, etiquetas)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(modelo_experto.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        perdida_train += loss.item()

    # --- VALIDACIÓN ---
    modelo_experto.eval()
    perdida_val = 0.0
    todos_probs, todos_labels = [], []

    with torch.no_grad():
        for imagenes, etiquetas in val_loader:
            imagenes, etiquetas = imagenes.to(device, non_blocking=True), etiquetas.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", dtype=torch.float16, enabled=(device.type == "cuda")):
                salidas = modelo_experto(imagenes)
                perdida_val += criterion(salidas, etiquetas).item()
            todos_probs.append(torch.sigmoid(salidas).float().cpu())
            todos_labels.append(etiquetas.int().cpu())

    # Métricas y Umbrales
    todos_probs = torch.cat(todos_probs, dim=0)
    todos_labels = torch.cat(todos_labels, dim=0)
    
    # Optimización de umbrales por clase
    preds_binarias = torch.zeros_like(todos_labels)
    for i in range(len(enfermedades)):
        precision, recall, thresholds = precision_recall_curve(todos_labels[:, i], todos_probs[:, i])
        f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
        best_thresh = float(thresholds[np.argmax(f1_scores)]) if len(thresholds) > 0 else 0.5
        preds_binarias[:, i] = (todos_probs[:, i] >= best_thresh).int()

    f1_macro = f1_score(todos_labels.numpy(), preds_binarias.numpy(), average="macro", zero_division=0)
    
    print(f"Época [{epoch+1}] Train Loss: {perdida_train/len(train_loader):.4f} | Val Loss: {perdida_val/len(val_loader):.4f} | F1 Macro: {f1_macro:.4f}")

    # Guardado y Early Stopping
    if f1_macro > mejor_f1:
        mejor_f1 = f1_macro
        epocas_sin_mejora = 0
        torch.save(modelo_experto.state_dict(), "mejor_experto_swin.pth")
        print(f"🌟 ¡Mejor modelo guardado! F1: {mejor_f1:.4f}")
    else:
        epocas_sin_mejora += 1
        if epocas_sin_mejora >= EARLY_STOPPING_PATIENCE:
            print("🛑 Early stopping alcanzado."); break

    scheduler.step()
    print("-" * 70)

Época 1/100 [Train]:   0%|          | 0/773 [00:00<?, ?it/s]

libpng warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG
libpng warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG


Época [1] Train Loss: 0.0288 | Val Loss: 0.0287 | F1 Macro: 0.4429
🌟 ¡Mejor modelo guardado! F1: 0.4429
----------------------------------------------------------------------


Época 2/100 [Train]:   0%|          | 0/773 [00:00<?, ?it/s]

libpng warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG
libpng warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG


...

Época 28/100 [Train]:   0%|          | 0/773 [00:00<?, ?it/s]

libpng warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG
libpng warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG


Época [28] Train Loss: 0.0224 | Val Loss: 0.0273 | F1 Macro: 0.5176
----------------------------------------------------------------------


Época 29/100 [Train]:   0%|          | 0/773 [00:00<?, ?it/s]

KeyboardInterrupt: 